# Rooftop Segmentation — BTP Demo

**Two-stage pipeline:**
1. U-Net (ResNet-34) rooftop segmentation from aerial RGB imagery
2. Area estimation: pixel count × GSD² → rooftop area (m²), estimated solar capacity (kW)

**Dataset:** AIRS — 7.5cm/pixel RGB aerial, Christchurch NZ  
**Target baseline:** PSPNet IoU=0.899 (Chen et al., 2019, ISPRS)

---
Run cells top-to-bottom. Step 1 installs deps. Step 2 mounts Drive. Step 3 trains. Step 4 demos inference.

## Step 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch Runtime > Change runtime type > GPU')

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations rasterio gdown opencv-python-headless tqdm

## Step 2 — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit these paths to match your Drive structure ──
DRIVE_ROOT   = '/content/drive/MyDrive/rooftop_solar'
AIRS_TRAIN   = os.path.join(DRIVE_ROOT, 'datasets/airs/train')
AIRS_VAL     = os.path.join(DRIVE_ROOT, 'datasets/airs/val')
AIRS_TEST    = os.path.join(DRIVE_ROOT, 'datasets/airs/test')
CKPT_DIR     = os.path.join(DRIVE_ROOT, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

# Verify structure
for path in [AIRS_TRAIN, AIRS_VAL, AIRS_TEST]:
    imgs = os.path.join(path, 'images')
    masks = os.path.join(path, 'masks')
    if os.path.exists(imgs):
        n_imgs = len(os.listdir(imgs))
        n_masks = len(os.listdir(masks)) if os.path.exists(masks) else 0
        print(f'{path.split("/")[-1]:6s}: {n_imgs} images, {n_masks} masks')
    else:
        print(f'MISSING: {imgs}')

## Step 3 — Configuration

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
CROP_SIZE       = 512       # px — crop size from 10k×10k tiles
OVERLAP_FRAC    = 0.1       # 10% overlap when tiling
BATCH_SIZE      = 8         # reduce to 4 if OOM on T4
NUM_EPOCHS      = 30        # 30 epochs ~45 min on T4 with 500 crops
LR              = 1e-4      # encoder LR; decoder gets 10× by default in SMP
ENCODER         = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
NUM_WORKERS     = 2
GSD_M           = 0.075     # meters per pixel for AIRS
MAX_TRAIN_CROPS = 2000      # cap for quick training; set None for full dataset
DEVICE          = 'cuda'    # falls back to cpu if no GPU

import torch
DEVICE = DEVICE if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 4 — Tile Cropping (10k×10k → 512×512)

In [ ]:
import numpy as np
import cv2
from pathlib import Path
from tqdm.auto import tqdm

def tile_image_mask(img_path, mask_path, crop_size=512, overlap=0.1, min_mask_frac=0.01):
    """Tile a large image+mask into overlapping crops. Skips near-empty mask crops."""
    img  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f'Mask not found: {mask_path}')

    # Binarize mask (AIRS masks are 0/255)
    mask = (mask > 127).astype(np.uint8) * 255

    H, W = img.shape[:2]
    stride = int(crop_size * (1 - overlap))
    crops = []

    for y in range(0, H - crop_size + 1, stride):
        for x in range(0, W - crop_size + 1, stride):
            img_crop  = img [y:y+crop_size, x:x+crop_size]
            mask_crop = mask[y:y+crop_size, x:x+crop_size]
            # Skip crops where almost no roof is visible
            if mask_crop.mean() / 255.0 >= min_mask_frac:
                crops.append((img_crop, mask_crop))

    return crops


def prepare_crops(split_dir, cache_dir, crop_size=512, overlap=0.1, max_crops=None):
    """Tile all images in a split and cache crops to disk."""
    img_dir  = Path(split_dir) / 'images'
    mask_dir = Path(split_dir) / 'masks'
    out_imgs  = Path(cache_dir) / 'images'
    out_masks = Path(cache_dir) / 'masks'
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_masks.mkdir(parents=True, exist_ok=True)

    # Check if already cached
    existing = list(out_imgs.glob('*.png'))
    if existing:
        print(f'  Using cached crops: {len(existing)} in {cache_dir}')
        if max_crops:
            return min(len(existing), max_crops)
        return len(existing)

    img_files = sorted(img_dir.glob('*.tif')) + sorted(img_dir.glob('*.png')) + sorted(img_dir.glob('*.jpg'))
    count = 0
    for img_path in tqdm(img_files, desc=f'Tiling {Path(split_dir).name}'):
        mask_path = mask_dir / (img_path.stem + '.png')
        if not mask_path.exists():
            mask_path = mask_dir / (img_path.stem + '.tif')
        if not mask_path.exists():
            print(f'  Skipping {img_path.name} — no mask')
            continue
        try:
            crops = tile_image_mask(img_path, mask_path, crop_size, overlap)
        except Exception as e:
            print(f'  Error {img_path.name}: {e}')
            continue
        for img_crop, mask_crop in crops:
            name = f'{img_path.stem}_{count:06d}'
            cv2.imwrite(str(out_imgs  / f'{name}.png'), cv2.cvtColor(img_crop, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_masks / f'{name}.png'), mask_crop)
            count += 1
            if max_crops and count >= max_crops:
                print(f'  Reached max_crops={max_crops}, stopping')
                return count
    print(f'  Tiled {count} crops from {len(img_files)} images')
    return count


# Cache tiled crops to /content/crops/ (local, fast I/O)
CROPS_TRAIN = '/content/crops/train'
CROPS_VAL   = '/content/crops/val'
CROPS_TEST  = '/content/crops/test'

print('Tiling training images...')
n_train = prepare_crops(AIRS_TRAIN, CROPS_TRAIN, CROP_SIZE, OVERLAP_FRAC, MAX_TRAIN_CROPS)
print(f'Train crops: {n_train}')

print('Tiling validation images...')
n_val = prepare_crops(AIRS_VAL, CROPS_VAL, CROP_SIZE, OVERLAP_FRAC, max_crops=500)
print(f'Val crops: {n_val}')

## Step 5 — Dataset & DataLoader

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader


def get_train_augmentation():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),        # 0/90/180/270° rotation
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=0, p=0.5),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.4),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_augmentation():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


class AIRSDataset(Dataset):
    def __init__(self, crops_dir, transform=None):
        self.img_dir  = Path(crops_dir) / 'images'
        self.mask_dir = Path(crops_dir) / 'masks'
        self.img_paths = sorted(self.img_dir.glob('*.png'))
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path  = self.img_paths[idx]
        mask_path = self.mask_dir / img_path.name

        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask  = (mask > 127).astype(np.float32)  # binary float [0,1]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask'].unsqueeze(0)  # [1, H, W]

        return image, mask


train_dataset = AIRSDataset(CROPS_TRAIN, transform=get_train_augmentation())
val_dataset   = AIRSDataset(CROPS_VAL,   transform=get_val_augmentation())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset)} crops | Val: {len(val_dataset)} crops')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Step 6 — Model, Loss, Metrics

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

# ── Model ────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=1,
    activation=None,   # raw logits — loss handles sigmoid
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: U-Net (ResNet-34) | Params: {total_params/1e6:.1f}M')


# ── Combined BCE + Dice Loss ──────────────────────────────────
bce_loss  = smp.losses.SoftBCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

def combined_loss(pred, target):
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)


# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(pred_logits, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred_logits) > threshold).float()
    tp = (pred_binary * target).sum()
    fp = (pred_binary * (1 - target)).sum()
    fn = ((1 - pred_binary) * target).sum()
    tn = ((1 - pred_binary) * (1 - target)).sum()

    iou       = tp / (tp + fp + fn + 1e-7)
    precision = tp / (tp + fp + 1e-7)
    recall    = tp / (tp + fn + 1e-7)
    f1        = 2 * precision * recall / (precision + recall + 1e-7)
    return {'iou': iou.item(), 'f1': f1.item(),
            'precision': precision.item(), 'recall': recall.item()}


# ── Optimizer ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': LR * 0.1},   # pretrained encoder: lower LR
    {'params': model.decoder.parameters(), 'lr': LR},
    {'params': model.segmentation_head.parameters(), 'lr': LR},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f'Optimizer: AdamW | Encoder LR: {LR*0.1:.1e} | Decoder LR: {LR:.1e}')

## Step 7 — Training Loop

In [ ]:
import time

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))  # mixed precision

best_val_iou = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_f1': []}

CKPT_PATH = os.path.join(CKPT_DIR, f'unet_resnet34_best.pth')

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    model.train()
    train_losses = []
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            preds = model(imgs)
            loss  = combined_loss(preds, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    # ── Validate ──
    model.eval()
    val_losses, val_metrics = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                preds = model(imgs)
                loss  = combined_loss(preds, masks)
            val_losses.append(loss.item())
            val_metrics.append(compute_metrics(preds, masks))

    scheduler.step()

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    val_iou    = np.mean([m['iou'] for m in val_metrics])
    val_f1     = np.mean([m['f1']  for m in val_metrics])
    val_prec   = np.mean([m['precision'] for m in val_metrics])
    val_rec    = np.mean([m['recall']    for m in val_metrics])
    elapsed    = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_f1'].append(val_f1)

    # Save best checkpoint to Drive
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'val_iou': val_iou, 'val_f1': val_f1}, CKPT_PATH)
        ckpt_marker = ' ← best'
    else:
        ckpt_marker = ''

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Loss {train_loss:.4f}/{val_loss:.4f} | '
          f'IoU {val_iou:.4f} | F1 {val_f1:.4f} | '
          f'P {val_prec:.4f} R {val_rec:.4f} | '
          f'{elapsed:.0f}s{ckpt_marker}')

print(f'\nBest val IoU: {best_val_iou:.4f} (saved to {CKPT_PATH})')
print(f'Baseline to beat: PSPNet IoU=0.899')

## Step 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'],   label='Val Loss')
ax1.set_title('Loss Curves')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_iou'], label='Val IoU', color='steelblue')
ax2.plot(history['val_f1'],  label='Val F1',  color='orange')
ax2.axhline(0.899, color='red', linestyle='--', label='PSPNet baseline IoU=0.899')
ax2.axhline(0.947, color='darkred', linestyle=':', label='PSPNet baseline F1=0.947')
ax2.set_title('Validation Metrics')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'logs/training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Load Best Model & Run Demo Inference

In [ ]:
# Load best checkpoint
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} | Val IoU: {checkpoint['val_iou']:.4f}")

# ── GSD-based area estimation ────────────────────────────────
SOLAR_EFFICIENCY_W_PER_M2 = 150  # typical residential panel

def estimate_area_and_capacity(mask_binary, gsd_m=GSD_M):
    """Convert binary mask to area (m²) and solar capacity (kW)."""
    pixel_area_m2 = gsd_m ** 2
    roof_pixels   = int(mask_binary.sum())
    roof_area_m2  = roof_pixels * pixel_area_m2
    capacity_kw   = roof_area_m2 * SOLAR_EFFICIENCY_W_PER_M2 / 1000
    return roof_area_m2, capacity_kw, roof_pixels


# ── Inference on a single crop ────────────────────────────────
def infer_single(img_path, mask_path=None):
    """Run inference on a single 512×512 image. Returns image, pred_mask, metrics_dict."""
    image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

    # Preprocess
    transform = get_val_augmentation()
    aug = transform(image=image, mask=np.zeros((image.shape[0], image.shape[1]), dtype=np.float32))
    img_tensor = aug['image'].unsqueeze(0).to(DEVICE)

    # Infer
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(img_tensor)
    pred_mask = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8)

    # Metrics (if GT mask provided)
    metrics = None
    if mask_path and Path(mask_path).exists():
        gt_mask = (cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        pred_t  = torch.tensor(pred_mask).float().unsqueeze(0).unsqueeze(0)
        gt_t    = torch.tensor(gt_mask).float().unsqueeze(0).unsqueeze(0)
        metrics = compute_metrics(pred_t * 10 - 5, gt_t)  # fake logits from binary

    area_m2, cap_kw, n_pixels = estimate_area_and_capacity(pred_mask)

    return image, pred_mask, metrics, area_m2, cap_kw


print('Model ready for inference.')

## Step 10 — Full Demo: Upload Image → Rooftop Mask → Area Estimate

In [ ]:
# ── Demo on sample test images ────────────────────────────────
test_img_dir  = Path(CROPS_TEST) / 'images'
test_mask_dir = Path(CROPS_TEST) / 'masks'

# Tile test images if not already done
if not test_img_dir.exists() or not any(test_img_dir.glob('*.png')):
    print('Tiling test images...')
    prepare_crops(AIRS_TEST, CROPS_TEST, CROP_SIZE, OVERLAP_FRAC, max_crops=200)

test_images = sorted(test_img_dir.glob('*.png'))[:6]  # show 6 samples
print(f'Running inference on {len(test_images)} test crops...')

# ── Visualization ────────────────────────────────────────────
fig, axes = plt.subplots(len(test_images), 4, figsize=(20, 5 * len(test_images)))
if len(test_images) == 1:
    axes = axes[np.newaxis, :]

OVERLAY_ALPHA = 0.4
ROOF_COLOR    = np.array([255, 100, 0], dtype=np.uint8)   # orange

all_metrics = []

for i, img_path in enumerate(test_images):
    mask_path = test_mask_dir / img_path.name
    image, pred_mask, metrics, area_m2, cap_kw = infer_single(img_path, mask_path)

    # Create overlay
    overlay = image.copy()
    overlay[pred_mask == 1] = (
        overlay[pred_mask == 1] * (1 - OVERLAY_ALPHA) +
        ROOF_COLOR * OVERLAY_ALPHA
    ).astype(np.uint8)

    # Boundary only visualization
    kernel = np.ones((3, 3), np.uint8)
    boundary = cv2.dilate(pred_mask, kernel) - cv2.erode(pred_mask, kernel)
    boundary_vis = image.copy()
    boundary_vis[boundary == 1] = [255, 0, 0]

    # GT mask if available
    gt_mask_vis = np.zeros_like(image)
    if mask_path.exists():
        gt = (cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        gt_mask_vis[gt == 1] = [0, 200, 0]

    axes[i, 0].imshow(image);           axes[i, 0].set_title('Input Image')
    axes[i, 1].imshow(gt_mask_vis);     axes[i, 1].set_title('Ground Truth')
    axes[i, 2].imshow(overlay);         axes[i, 2].set_title(
        f'Predicted Mask\nArea: {area_m2:.1f}m² | Cap: {cap_kw:.2f}kW')
    axes[i, 3].imshow(boundary_vis);    axes[i, 3].set_title(
        f'Boundary | IoU: {metrics["iou"]:.3f} F1: {metrics["f1"]:.3f}'
        if metrics else 'Boundary')

    for ax in axes[i]: ax.axis('off')
    if metrics:
        all_metrics.append(metrics)

plt.suptitle('Rooftop Segmentation Demo — U-Net (ResNet-34)', fontsize=16, y=1.01)
plt.tight_layout()
save_path = os.path.join(DRIVE_ROOT, 'logs/demo_results.png')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

if all_metrics:
    print(f'\n=== Demo Summary ===')
    print(f'Mean IoU:       {np.mean([m["iou"] for m in all_metrics]):.4f}  (PSPNet baseline: 0.899)')
    print(f'Mean F1:        {np.mean([m["f1"]  for m in all_metrics]):.4f}  (PSPNet baseline: 0.947)')
    print(f'Mean Precision: {np.mean([m["precision"] for m in all_metrics]):.4f}')
    print(f'Mean Recall:    {np.mean([m["recall"]    for m in all_metrics]):.4f}')
    print(f'Saved to: {save_path}')

## Step 11 — Upload Your Own Aerial Image (Interactive)

In [ ]:
from google.colab import files
import io
from PIL import Image

print('Upload an aerial image (512×512 px, RGB). Larger images will be center-cropped.')
uploaded = files.upload()

for fname, data in uploaded.items():
    # Load and resize
    pil_img = Image.open(io.BytesIO(data)).convert('RGB')
    # Center-crop to CROP_SIZE
    w, h = pil_img.size
    if w != CROP_SIZE or h != CROP_SIZE:
        # Crop from center
        left = (w - CROP_SIZE) // 2
        top  = (h - CROP_SIZE) // 2
        pil_img = pil_img.crop((left, top, left + CROP_SIZE, top + CROP_SIZE))
        print(f'Center-cropped from {w}×{h} to {CROP_SIZE}×{CROP_SIZE}')

    image = np.array(pil_img)

    # Preprocess & infer
    transform = get_val_augmentation()
    aug = transform(image=image, mask=np.zeros((CROP_SIZE, CROP_SIZE), dtype=np.float32))
    img_tensor = aug['image'].unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
    pred_mask = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8)

    area_m2, cap_kw, n_pixels = estimate_area_and_capacity(pred_mask)

    # Visualize
    overlay = image.copy()
    overlay[pred_mask == 1] = (overlay[pred_mask == 1] * 0.6 + ROOF_COLOR * 0.4).astype(np.uint8)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    ax1.imshow(image);          ax1.set_title('Input Image',    fontsize=13)
    ax2.imshow(pred_mask, cmap='gray'); ax2.set_title('Predicted Mask', fontsize=13)
    ax3.imshow(overlay);        ax3.set_title(
        f'Overlay\nRooftop: {area_m2:.1f} m²  |  Est. capacity: {cap_kw:.2f} kW', fontsize=13)
    for ax in (ax1, ax2, ax3): ax.axis('off')
    plt.suptitle(f'Rooftop Segmentation — {fname}', fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f'\nRooftop pixels:  {n_pixels:,}')
    print(f'Rooftop area:    {area_m2:.2f} m²  (GSD = {GSD_M}m/pixel)')
    print(f'Est. capacity:   {cap_kw:.2f} kW  (assuming {SOLAR_EFFICIENCY_W_PER_M2} W/m²)')

## Step 12 — Full Test-Set Evaluation

In [ ]:
# Run on all test crops
print('Tiling test images...')
prepare_crops(AIRS_TEST, CROPS_TEST, CROP_SIZE, OVERLAP_FRAC)

test_dataset = AIRSDataset(CROPS_TEST, transform=get_val_augmentation())
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
print(f'Test crops: {len(test_dataset)}')

model.eval()
test_metrics = []
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Evaluating test set'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            preds = model(imgs)
        test_metrics.append(compute_metrics(preds, masks))

print('\n=== Test Set Results ===')
print(f'IoU:       {np.mean([m["iou"] for m in test_metrics]):.4f}   ← target: > 0.899 (PSPNet)')
print(f'F1:        {np.mean([m["f1"]  for m in test_metrics]):.4f}   ← target: > 0.947 (PSPNet)')
print(f'Precision: {np.mean([m["precision"] for m in test_metrics]):.4f}')
print(f'Recall:    {np.mean([m["recall"]    for m in test_metrics]):.4f}')